# GraphRAG Under Community Partitions — Fast Modular Pipeline

This notebook runs the MP2 GraphRAG experiment on Kaggle with an optimized, modular 4-step pipeline:

| Step | Action | GPU Needed? | Time (40 Graphs) |
| :--- | :--- | :---: | :--- |
| **Step 1: Prepare** | SBM generation, Leiden clustering, cut strata, QA generation | No | ~5 seconds |
| **Step 2: Summarize** | Ollama LLM summaries (18 calls/graph across 3 prompt conditions) | **Yes (T4)** | ~2 hours (vs 18+ hrs originally!) |
| **Step 3: Evaluate** | Compute retention metrics, DiD lift, and Oracle QA accuracy | No | ~3 seconds |
| **Step 4: Report** | Statistical aggregation (t-tests, Holm correction) & `summary.md` | No | ~1 second |

> **Why is this 7x faster?**
> The original pipeline ran 120 sequential LLM answering calls per graph on top of 18 summary calls. Step 3 replaces answering noise with exact deterministic Oracle QA, completing the evaluation in seconds while preserving 100% of the statistical hypothesis tests.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Install Ollama & dependencies
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # wait for daemon to start

# Pull model (qwen3:8b is recommended; qwen3:4b or qwen3:1.7b can be used for extra speed)
!ollama pull qwen3:8b

In [ ]:
# Cell 3: Clone or update repo
import os
if not os.path.exists('/kaggle/working/mp2'):
    !git clone https://github.com/Bvijay03/GraphRag.git /kaggle/working/mp2
%cd /kaggle/working/mp2
!git pull

# Install Python dependencies
!pip install -q igraph leidenalg scipy statsmodels pyyaml requests

In [ ]:
# Cell 4: Verify Ollama LLM is responding
from llm import OllamaLLM
llm = OllamaLLM(model="qwen3:8b", num_ctx=4096, temperature=0, seed=42, think=False)
res = llm.generate("Respond in one word: ready?")
print("LLM status:", res.text.strip())

### Step 1: Prepare All Graphs (Takes ~5 seconds)
Builds the SBM graphs, runs Leiden hierarchical clustering, stratifies triples (within vs cut), and generates multi-hop evaluation questions.

In [ ]:
# Cell 5: Step 1 (Prepare)
!python step1_prepare.py --config config.yaml --out runs/kaggle_exp

### Step 2: Run LLM Summaries (GPU Accelerated)
Generates community summaries under 3 conditions:
1. `default`: Standard GraphRAG prompt
2. `topology`: Topologically marked prompt
3. `control`: Randomly marked control prompt

*Note: Summaries are saved immediately per graph. If disconnected, simply re-run this cell to resume!*

In [ ]:
# Cell 6: Step 2 (Summarize)
!python step2_summarize.py --config config.yaml --out runs/kaggle_exp --backend ollama

### Step 3: Evaluate Retention & Multi-Hop QA (Takes ~3 seconds)
Combines the prepared graphs with the generated summaries to compute L1/L2 retention, stratum breakdown (within vs cut), intervention DiD, and QA accuracy.

In [ ]:
# Cell 7: Step 3 (Evaluate)
!python step3_evaluate.py --config config.yaml --out runs/kaggle_exp

### Step 4: Aggregate Statistical Report & Results
Performs t-tests, Holm family-wise error correction, and writes `summary.md` and `results.json`.

In [ ]:
# Cell 8: Step 4 (Report)
!python step4_report.py --config config.yaml --out runs/kaggle_exp

# Display the markdown report directly in the notebook
from IPython.display import Markdown, display
display(Markdown(filename='runs/kaggle_exp/summary.md'))

### Diagnostic Inspector: Inspect Prompts, Summaries & QA Errors
Use `inspect_run.py` to examine any graph, community prompt, or QA failure to see where the LLM dropped facts.

In [ ]:
# Cell 9: Run overview
!python inspect_run.py --dir runs/kaggle_exp

# Cell 9b: Inspect QA errors for seed 1 (optional)
# !python inspect_run.py --dir runs/kaggle_exp --seed 1 --qa-errors

### Download Results
Zip the outputs for downloading to your local machine.

In [ ]:
# Cell 10: Zip results
!zip -r /kaggle/working/mp2_results.zip runs/kaggle_exp/
print('Ready to download from /kaggle/working/mp2_results.zip')